In [3]:
import pandas as pd
import numpy as np

ruta_origen = '00_locatel0311-completa.csv'
ruta_destino = 'sipa_etl_procesado.csv'

# 1. Extracción con optimización de memoria
columnas_base = [
    'id_folio', 'fecha_solicitud', 'tema_solicitud', 'fecha_concluido', 
    'colonia_datos', 'alcaldia', 'estatus', 'latitud', 'longitud'
]
tipos_datos = {
    'id_folio': 'str', 'tema_solicitud': 'str', 'colonia_datos': 'str', 
    'alcaldia': 'str', 'estatus': 'str', 'fecha_concluido': 'str'
}

print("Cargando dataset crudo...")
df = pd.read_csv(ruta_origen, usecols=lambda c: c in columnas_base, dtype=tipos_datos, low_memory=False)

# 2. Transformación y Filtrado
print("Filtrando reportes de bacheo...")
df = df[df['tema_solicitud'].str.contains('BACHE', case=False, na=False)].copy()

print("Limpiando folios, coordenadas y nulos...")
df['id_folio'] = df['id_folio'].str.replace('-', '', regex=False).str.strip()

df['latitud'] = pd.to_numeric(df['latitud'], errors='coerce')
df['longitud'] = pd.to_numeric(df['longitud'], errors='coerce')
df = df.dropna(subset=['latitud', 'longitud'])
df = df[(df['latitud'] != 0) & (df['longitud'] != 0)]

print("Estandarizando texto y fechas...")
# Quitamos el formato estricto para que Pandas lea el YYYY-MM-DD original
df['fecha_solicitud'] = pd.to_datetime(df['fecha_solicitud'], errors='coerce')
df['fecha_concluido'] = pd.to_datetime(df['fecha_concluido'], errors='coerce')

columnas_texto = ['colonia_datos', 'alcaldia', 'estatus']
for col in columnas_texto:
    df[col] = df[col].str.strip().str.upper()

# 3. Ingeniería de Características (Geo-Hashing)
print("Generando identificador geoespacial...")
df['celda_geo'] = df['latitud'].round(4).astype(str) + "_" + df['longitud'].round(4).astype(str)

# 4. Auditoría y Exportación
df = df.drop(columns=['tema_solicitud']) 
# Solo borramos si no hay fecha de solicitud, dejamos las alcaldías en paz
df = df.dropna(subset=['fecha_solicitud']) 
df = df.drop_duplicates(subset=['id_folio']) 

print(f"ETL completado. Registros finales listos para SQL: {len(df)}")
df.to_csv(ruta_destino, index=False)

Cargando dataset crudo...
Filtrando reportes de bacheo...
Limpiando folios, coordenadas y nulos...
Estandarizando texto y fechas...
Generando identificador geoespacial...
ETL completado. Registros finales listos para SQL: 35743


In [4]:
import pandas as pd

# Leer solo la columna de temas del archivo crudo
df_auditoria = pd.read_csv('00_locatel0311-completa.csv', usecols=['tema_solicitud'], low_memory=False)

# Aplicar exactamente el mismo filtro del ETL
filtro_baches = df_auditoria[df_auditoria['tema_solicitud'].str.contains('BACHE', case=False, na=False)]

# Mostrar todas las categorías únicas que atrapamos
categorias_atrapadas = filtro_baches['tema_solicitud'].unique()

print("Categorías que estamos considerando como baches:")
for categoria in categorias_atrapadas:
    print(f"- {categoria}")

Categorías que estamos considerando como baches:
- BACHEO


In [5]:
import pandas as pd
from sqlalchemy import create_engine
import urllib

# 1. Cargar el dataset procesado
ruta_csv = 'sipa_etl_procesado.csv'
df_limpio = pd.read_csv(ruta_csv)

# 2. Configurar la cadena de conexión a SQL Server
nombre_servidor = 'localhost' 
base_datos = 'SIPA_Portfolio'

parametros_conexion = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={nombre_servidor};"
    f"DATABASE={base_datos};"
    "Trusted_Connection=yes;"
)

# 3. Crear el motor con inyección rápida por lotes
motor_sql = create_engine(f"mssql+pyodbc:///?odbc_connect={parametros_conexion}", fast_executemany=True)

# 4. Ejecutar la inyección masiva
print(f"Iniciando inyección de {len(df_limpio)} registros hacia SQL Server...")
df_limpio.to_sql('stg_reportes_bacheo', con=motor_sql, if_exists='append', index=False)
print("¡Operación completada exitosamente!")

Iniciando inyección de 35743 registros hacia SQL Server...
¡Operación completada exitosamente!
